In [ ]:
# !pip install matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
distributions = [
    {"name": "простой",          "dist": [0.02, 0.10, 0.38, 0.50]},
    {"name": "сложный",          "dist": [0.25, 0.40, 0.25, 0.10]},
    {"name": "50/50 смесь",           "dist": [0.135, 0.25, 0.315, 0.30]},
    {"name": "90% простой + 10% сложный",      "dist": [0.043, 0.13, 0.367, 0.46]},
    {"name": "70% простой + 30% сложный",      "dist": [0.089, 0.19, 0.341, 0.38]},
    {"name": "30% простой + 70% сложный",      "dist": [0.181, 0.31, 0.289, 0.22]},
    {"name": "10% простой + 90% сложный",      "dist": [0.227, 0.37, 0.263, 0.14]},
    {"name": "все двойки",                     "dist": [1.0,  0.0,  0.0,  0.0]},
    {"name": "все пятёрки",                    "dist": [0.0,  0.0,  0.0,  1.0]},
    {"name": "все тройки",                     "dist": [0.0,  1.0,  0.0,  0.0]},
    {"name": "все четвёрки",                   "dist": [0.0,  0.0,  1.0,  0.0]},
    {"name": "равномерное 25/25/25/25",       "dist": [0.25, 0.25, 0.25, 0.25]},
    {"name": "бимодальное: только 2 и 5",      "dist": [0.5,  0.0,  0.0,  0.5]},
    {"name": "бимодальное: 3 и 4",             "dist": [0.0,  0.5,  0.5,  0.0]},
    {"name": "провал в центре (2,5)",          "dist": [0.4,  0.1,  0.1,  0.4]},
    {"name": "нормальное ~тройка",             "dist": [0.10, 0.40, 0.40, 0.10]},
    {"name": "нормальное ~четвёрка",           "dist": [0.05, 0.20, 0.50, 0.25]},
    {"name": "много троек, мало пятёрок",      "dist": [0.15, 0.50, 0.25, 0.10]},
    {"name": "много четвёрок, мало двоек",     "dist": [0.05, 0.15, 0.60, 0.20]},
    {"name": "много пятёрок, есть двойки",     "dist": [0.15, 0.10, 0.25, 0.50]},
    {"name": "нули в двойках и пятёрках",      "dist": [0.0,  0.5,  0.5,  0.0]},
    {"name": "нули в тройках и пятёрках",      "dist": [0.3,  0.0,  0.7,  0.0]},
]



## Тест 1

Давайте попробуем взять два эталонных распределения, простого и сложного предмета и попробуем оценить их сложность через критерий хи-квадрат

Исходя из исходных данных я выберу эталонные распределения такими
- `d_ez = [0.01, 0.11, 0.27, 0.61]`
- `d_hd = [0.1, 0.58, 0.27, 0.15]`

Теперь пусть у нас есть распредление `d_ec` и мы хотим понять, сложный предмет или нет

Давайте посчитаем

$$
\begin{align*}
\text{dist}_{ez} = \chi^2(d_{ec}, d_{ez})\\
\text{dist}_{hd} = \chi^2(d_{ec}, d_{hd})
\end{align*}
$$

Тогда сложность можно вычислять, как 
$$
diff = \frac{\text{dist}_{ez}}{\text{dist}_{ez} + \text{dist}_{hd}}
$$


In [ ]:
d_ez = [0.03, 0.10, 0.26, 0.61]
d_hd = [0.15, 0.57, 0.24, 0.14]


In [ ]:
def chi_2_2(d_ec, d_ez, d_hd):
    d_ec = np.asarray(d_ec, dtype=float)
    d_ez = np.asarray(d_ez, dtype=float)
    d_hd = np.asarray(d_hd, dtype=float)

    dist_to_easy = np.sum((d_ec - d_ez) ** 2 / d_ez)
    dist_to_hard = np.sum((d_ec - d_hd) ** 2 / d_hd)

    return dist_to_easy / (dist_to_easy + dist_to_hard)

In [ ]:
def tests_chi_2_2(distributions, d_ez, d_hd):
    colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
    for d in distributions:
        score = chi_2_2(d["dist"], d_ez, d_hd)
        print(f"\n{d['name']}")
        print(f"  Сложность: {score:.4f}")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for ax, (title, dist) in zip(axes, [
            ("Простой эталон", d_ez),
            (f"Реальное: {d['name']}", d["dist"]),
            ("Сложный эталон", d_hd)
        ]):
            bars = ax.bar(["2", "3", "4", "5"], dist, color=colors, edgecolor="white", linewidth=1.2)
            ax.set_title(title, fontsize=11)
            ax.set_ylim(0, 1)
            ax.set_ylabel("Доля")
            for bar, val in zip(bars, dist):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f"{val:.2f}", ha="center", fontsize=9)
        
        plt.suptitle(f"Сложность: {score:.3f}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
tests_chi_2_2(distributions, d_ez, d_hd)

## Возможные проблемы

Для предмета, у которого все двойки или все пятёрки, такой метод выдаёт сложность 0.849 и 0.1 соответсвенно, хотелось бы это как-то поправить, также не совсем понятно, хорошо или плохо то, что при равномерном распределении тест выдаёт 0.86, то есть считает предмет сложным, причём более сложным, чем предмет, по которому пятёрок нет вообще

## Тест 2

Попробуем исправить это при помощи среднего

In [ ]:
def chi_2_2_mean(d_ec, d_ez, d_hd, w_shape=0.6):
    d_ec = np.asarray(d_ec, dtype=float)
    d_ez = np.asarray(d_ez, dtype=float)
    d_hd = np.asarray(d_hd, dtype=float)
    mean = d_ec.mean()
    dist_to_easy = np.sum((d_ec - d_ez) ** 2 / d_ez)
    dist_to_hard = np.sum((d_ec - d_hd) ** 2 / d_hd)

    chi_2 = dist_to_easy / (dist_to_easy + dist_to_hard)
    
    grades = np.array([2, 3, 4, 5])
    mean_score = np.dot(d_ec, grades)
    s_mean = 1.0 - (mean_score - 2.0) / 3.0
    s_mean = np.clip(s_mean, 0.0, 1.0)

    return w_shape * chi_2 + (1 - w_shape) * s_mean

In [ ]:
def tests_chi_2_2_mean(distributions, d_ez, d_hd, w_shape=0.5):
    colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
    for d in distributions:
        score = chi_2_2_mean(d["dist"], d_ez, d_hd, w_shape)
        print(f"\n{d['name']}")
        print(f"  Сложность: {score:.4f}")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for ax, (title, dist) in zip(axes, [
            ("Простой эталон", d_ez),
            (f"Реальное: {d['name']}", d["dist"]),
            ("Сложный эталон", d_hd)
        ]):
            bars = ax.bar(["2", "3", "4", "5"], dist, color=colors, edgecolor="white", linewidth=1.2)
            ax.set_title(title, fontsize=11)
            ax.set_ylim(0, 1)
            ax.set_ylabel("Доля")
            for bar, val in zip(bars, dist):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f"{val:.2f}", ha="center", fontsize=9)
        
        plt.suptitle(f"Сложность: {score:.3f}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
tests_chi_2_2_mean(distributions, d_ez, d_hd, 0.3)

In [ ]:
tests_chi_2_2_mean(distributions, d_ez, d_hd, 0.5)

In [ ]:
tests_chi_2_2_mean(distributions, d_ez, d_hd, 0.7)

## Итог

Ситуация улучшилась, получилось сделать более хорошие слоджности в вырожденых случаях

## Тест 3


In [ ]:
res = 0
for m2 in np.linspace(0, 1, 50):
    for m3 in np.linspace(0, 1, 50):
        for m4 in np.linspace(0, 1, 50):
            m5 = 1 - m2 - m3 - m4
            if m5 < 0:
                continue
            for d in distributions:
                res = max(res, chi_2_2(d["dist"], d_ez, d_hd))
print(res)

In [ ]:
res = 10**9
for m2 in np.linspace(0, 1, 50):
    for m3 in np.linspace(0, 1, 50):
        for m4 in np.linspace(0, 1, 50):
            m5 = 1 - m2 - m3 - m4
            if m5 < 0:
                continue
            for d in distributions:
                res = min(res, chi_2_2(d["dist"], d_ez, d_hd))
print(res)

In [ ]:
max_eps = 0.9919113765151001

In [ ]:
def chi_2_2_eps(d_ec, d_ez, d_hd):
    d_ec = np.asarray(d_ec, dtype=float)
    d_ez = np.asarray(d_ez, dtype=float)
    d_hd = np.asarray(d_hd, dtype=float)

    dist_to_easy = np.sum((d_ec - d_ez) ** 2 / d_ez)
    dist_to_hard = np.sum((d_ec - d_hd) ** 2 / d_hd)

    return dist_to_easy / (dist_to_easy + dist_to_hard) / max_eps

In [ ]:
def tests_chi_2_2_eps(distributions, d_ez, d_hd):
    colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
    for d in distributions:
        score = chi_2_2_eps(d["dist"], d_ez, d_hd)
        print(f"\n{d['name']}")
        print(f"  Сложность: {score:.4f}")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for ax, (title, dist) in zip(axes, [
            ("Простой эталон", d_ez),
            (f"Реальное: {d['name']}", d["dist"]),
            ("Сложный эталон", d_hd)
        ]):
            bars = ax.bar(["2", "3", "4", "5"], dist, color=colors, edgecolor="white", linewidth=1.2)
            ax.set_title(title, fontsize=11)
            ax.set_ylim(0, 1)
            ax.set_ylabel("Доля")
            for bar, val in zip(bars, dist):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f"{val:.2f}", ha="center", fontsize=9)
        
        plt.suptitle(f"Сложность: {score:.3f}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
tests_chi_2_2_eps(distributions, d_ez, d_hd)

In [ ]:
def chi_2_2_arctan(d_ec, d_ez, d_hd, strength=10.0):
    raw = chi_2_2(d_ec, d_ez, d_hd)
    t = strength * (raw - 0.5)
    return 0.5 + np.arctan(t) / np.pi

In [ ]:
def tests_chi_2_2_arctan(distributions, d_ez, d_hd):
    colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
    for d in distributions:
        score = chi_2_2_arctan(d["dist"], d_ez, d_hd)
        print(f"\n{d['name']}")
        print(f"  Сложность: {score:.4f}")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        for ax, (title, dist) in zip(axes, [
            ("Простой эталон", d_ez),
            (f"Реальное: {d['name']}", d["dist"]),
            ("Сложный эталон", d_hd)
        ]):
            bars = ax.bar(["2", "3", "4", "5"], dist, color=colors, edgecolor="white", linewidth=1.2)
            ax.set_title(title, fontsize=11)
            ax.set_ylim(0, 1)
            ax.set_ylabel("Доля")
            for bar, val in zip(bars, dist):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f"{val:.2f}", ha="center", fontsize=9)
        
        plt.suptitle(f"Сложность: {score:.3f}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
tests_chi_2_2_arctan(distributions, d_ez, d_hd)